# 03 · Market Basket — what sells together
Apriori association rules over `pos_checks.items` (food ↔ wine). Production computes pair lift in `engine/association.ts`; this notebook scales to itemsets ≥2 and mines directional rules with confidence.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from wineops_data import get_checks, get_tables, get_consumption, get_orders, get_inventory, daily_series
plt.rcParams['figure.figsize'] = (11, 4)

checks = get_checks()
checks['items'] = checks['items'].apply(lambda x: x if isinstance(x, list) else [])
txns = [sorted({i['name'] for i in row}) for row in checks['items'] if len(row) >= 2]
print(len(txns), 'transactions')

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
te = TransactionEncoder()
X = pd.DataFrame(te.fit(txns).transform(txns), columns=te.columns_)
freq = apriori(X, min_support=0.01, use_colnames=True)
rules = association_rules(freq, metric='lift', min_threshold=1.2)
rules['antecedents'] = rules['antecedents'].apply(lambda s: ', '.join(s))
rules['consequents'] = rules['consequents'].apply(lambda s: ', '.join(s))
rules.sort_values('lift', ascending=False).head(12)[['antecedents','consequents','support','confidence','lift']]

In [ ]:
# Food → wine rules only: the actionable menu prompts
wine_names = {i['name'] for row in checks['items'] for i in row if i.get('is_wine')}
f2w = rules[rules['consequents'].isin(wine_names) & ~rules['antecedents'].isin(wine_names)]
f2w.sort_values('lift', ascending=False).head(8)[['antecedents','consequents','confidence','lift']]

In [ ]:
top = f2w.sort_values('lift', ascending=False).head(8)
plt.barh(top['antecedents'] + ' → ' + top['consequents'], top['lift'])
plt.axvline(1, color='k', ls='--'); plt.title('Food → wine lift (>1 = affinity)'); plt.tight_layout(); plt.show()

**Action:** rules with lift > 1.5 and confidence > 30% are print-on-menu pairings; feed the winners into `pos_item_mappings` + server scripts, then A/B attach rate via `/analytics/table-performance`.